# ATML PA3 Task 3 on Kaggle

1. Import this notebook into Kaggle. Enable **Internet** and select **GPU T4** (or **GPU T4 ×2**). The notebook exposes only one T4 to the worker.
2. Select **Save Version → Save & Run All** with the GPU enabled. For an interactive run, save the session outputs before stopping it.
3. If the run stops before completion, attach the saved version through **Add Input → Notebook Output Files**, then run the notebook again. Attach only one Task 3 output version at a time. The launcher automatically restores its files and the pipeline checks checkpoint identity and checksums.

Logs, outputs, and checkpoints are written under `/kaggle/working/ATML_PA3_Task3/`. Only a saved Kaggle output version can be restored in a new session. The default five-hour session budget limits each run. Total runtime has not been measured on Kaggle.


In [ ]:
from pathlib import Path
import datetime, hashlib, json, os, subprocess, sys, threading, time

ROOT = Path('/kaggle/working/ATML_PA3_Task3')
ROOT.mkdir(parents=True, exist_ok=True)
LOG = ROOT / 'full_pipeline.log'
_log_lock = threading.RLock()
def log(message):
    line = datetime.datetime.now(datetime.timezone.utc).isoformat() + ' ' + message
    with _log_lock:
        print(line, flush=True)
        with LOG.open('a') as f:
            f.write(line + '\n')
            f.flush()

def stream(command, cwd=None, env=None):
    log('[START] ' + ' '.join(map(str, command)))
    done = threading.Event()
    def heartbeat():
        while not done.wait(30):
            log('[HEARTBEAT] Command active: ' + ' '.join(map(str, command[:4])))
    thread = threading.Thread(target=heartbeat, daemon=True)
    thread.start()
    try:
        with subprocess.Popen(command, cwd=cwd, env=env, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1) as child:
            for line in child.stdout:
                log(line.rstrip())
            code = child.wait()
        if code:
            raise RuntimeError('Command failed with exit code ' + str(code) + ': ' + ' '.join(command))
    except BaseException as exc:
        log('[FAIL] ' + repr(exc))
        failed = {'state': 'failed', 'fully_completed': False, 'error': repr(exc)}
        for name in ('status.json', 'final_verification.json'):
            temp = ROOT / (name + '.partial')
            temp.write_text(json.dumps(failed, indent=2))
            temp.replace(ROOT / name)
        raise
    finally:
        done.set()
        thread.join(timeout=2)
    log('[PASS] Command finished; this is not proof of experiment completion')
(ROOT / 'status.json').write_text(json.dumps({'state': 'not run', 'fully_completed': False}))
log('[START] Task 3 launcher. Logs: ' + str(LOG))
import shutil

def restore_saved_output(root, input_dir):
    """Fill missing files from one attached Task 3 version, preserving current work."""
    candidates = []
    for current, directories, _ in os.walk(input_dir):
        folder = Path(current)
        if folder.name == root.name and any((folder / name).is_dir() for name in ('release', 'runs')):
            candidates.append(folder)
            directories[:] = []
        elif len(folder.relative_to(input_dir).parts) >= 7:
            directories[:] = []
    if len(candidates) > 1:
        raise RuntimeError('Attach only the Task 3 output version you want to resume.')
    if not candidates:
        log('[START] No attached Task 3 output version found.')
        return
    source = candidates[0]
    log('[START] Restoring from ' + str(source))
    for name in ('release', 'runs', 'model_cache'):
        parent = source / name
        if not parent.is_dir():
            continue
        for current, directories, files in os.walk(parent):
            directories[:] = [directory for directory in directories
                              if directory not in ('staging', '.locks')]
            for filename in files:
                path = Path(current) / filename
                if filename.endswith(('.partial', '.incomplete', '.lock')) or not path.is_file():
                    continue
                target = root / path.relative_to(source)
                if target.exists():
                    continue
                target.parent.mkdir(parents=True, exist_ok=True)
                temporary = target.with_name(target.name + '.restore.partial')
                shutil.copyfile(path, temporary)
                temporary.replace(target)
    log('[PASS] Attached files restored. The worker will validate them before reuse.')

restore_saved_output(ROOT, Path('/kaggle/input'))


In [ ]:
SOURCE_REVISION = 'b370e572a01f5b0c3cd4d12e0195314be647093f'
REPOSITORY = 'https://github.com/Mujtaba7734/Advanced-Topics-in-Machine-Learning.git'
CODE = Path('/tmp/atml_task3_' + SOURCE_REVISION[:12])
CODE.mkdir(parents=True, exist_ok=True)
stream(['git', 'init', str(CODE)])
stream(['git', '-C', str(CODE), 'fetch', '--depth=1', '--filter=blob:none', REPOSITORY, SOURCE_REVISION])
stream(['git', '-C', str(CODE), 'sparse-checkout', 'set', '--no-cone',
        '/Assignment-3/task3/', '/Assignment-3/tests/test_task3.py', '/Assignment-3/ATML-PA3.pdf'])
stream(['git', '-C', str(CODE), 'checkout', '--detach', 'FETCH_HEAD'])
stream(['git', '-C', str(CODE), 'diff', '--exit-code', 'HEAD', '--', 'Assignment-3/task3', 'Assignment-3/tests/test_task3.py'])
ASSIGNMENT = CODE / 'Assignment-3'
ENV = dict(os.environ, PYTHONUNBUFFERED='1', PYTHONPATH=str(ASSIGNMENT),
           HF_HOME=str(ROOT / 'model_cache'), TASK3_NOTEBOOK_TEE='1', TOKENIZERS_PARALLELISM='false')
log('[PASS] Pinned source commit: ' + SOURCE_REVISION)
# Set device visibility before the subprocess imports CUDA PyTorch.
visible = os.environ.get('CUDA_VISIBLE_DEVICES', '0')
ENV['CUDA_VISIBLE_DEVICES'] = visible.split(',')[0].strip()
ENV['HF_HUB_DISABLE_XET'] = '1'
stream([sys.executable, '-u', '-c', """
import torch
if not torch.cuda.is_available() or torch.cuda.device_count() != 1:
    raise RuntimeError('Enable GPU T4 (or GPU T4 x2) in Kaggle settings.')
name = torch.cuda.get_device_name(0)
if 'T4' not in name:
    raise RuntimeError(f'Task 3 requires a T4. Detected {name}.')
print(f'[PASS] {name}, one visible GPU, installed PyTorch {torch.__version__}')
"""], cwd=str(ASSIGNMENT), env=ENV)
import importlib.metadata as metadata

missing = []
for requirement in (ASSIGNMENT / 'task3/requirements-colab.txt').read_text().splitlines():
    if not requirement or requirement.startswith('#'):
        continue
    package, version = requirement.split('==', 1)
    try:
        installed = metadata.version(package)
    except metadata.PackageNotFoundError:
        installed = None
    if installed != version:
        missing.append(requirement)
if missing:
    constraint = CODE / 'torch-constraint.txt'
    constraint.write_text('torch==' + metadata.version('torch') + '\n')
    stream([sys.executable, '-u', '-m', 'pip', 'install', '--disable-pip-version-check',
            '--prefer-binary', '-c', str(constraint), *missing], env=ENV)
else:
    log('[SKIP] All pinned dependencies are already installed.')
stream([sys.executable, '-u', '-m', 'pytest', '-q', 'tests/test_task3.py'],
       cwd=str(ASSIGNMENT), env=ENV)


In [ ]:
def run_task3():
    log('[START] Official offline experiments, then live T4 experiment. Existing verified results will be reused.')
    stream([sys.executable, '-u', '-m', 'task3.run', '--root', str(ROOT)], cwd=str(ASSIGNMENT), env=ENV)

def prepare_saved_output(root):
    """Materialize Hugging Face snapshot links without keeping duplicate model blobs."""
    from hashlib import file_digest

    model_cache = root / 'model_cache'
    for manifest_file in model_cache.glob('*-verified.json'):
        manifest = json.loads(manifest_file.read_text())
        model_dir = model_cache / ('models--' + manifest['model'].replace('/', '--'))
        snapshot = model_dir / 'snapshots' / manifest['revision']
        blob_dir = (model_dir / 'blobs').resolve()
        links = []
        for filename, expected in manifest['files'].items():
            path = snapshot / filename
            if not path.is_symlink():
                continue
            blob = path.resolve()
            if not blob.is_relative_to(blob_dir) or not blob.is_file():
                raise RuntimeError('Invalid cached model link: ' + str(path))
            with blob.open('rb') as handle:
                if file_digest(handle, 'sha256').hexdigest() != expected:
                    raise RuntimeError('Cached model checksum mismatch: ' + str(path))
            links.append((path, blob))
        for path, blob in links:
            temporary = path.with_name(path.name + '.portable.partial')
            temporary.unlink(missing_ok=True)
            os.link(blob, temporary)
            temporary.replace(path)
        for blob in {blob for _, blob in links}:
            # Keep blobs still referenced by any other snapshot in this cache.
            if not any(link.is_symlink() and link.resolve() == blob
                       for link in (model_dir / 'snapshots').rglob('*')):
                blob.unlink()
        if links:
            log('[PASS] Verified model snapshot prepared as regular Kaggle output files.')
    # These are reconstructible copies, never the release, exports, or checkpoints.
    for temporary in [root / 'local_ssd', *root.glob('runs/*/staging')]:
        if temporary.is_dir() and not temporary.is_symlink():
            shutil.rmtree(temporary)

try:
    run_task3()
finally:
    prepare_saved_output(ROOT)
    log('[KAGGLE OUTPUT] Save this version with output files before ending the session.')


In [ ]:
verification = ROOT / 'final_verification.json'
if not verification.is_file():
    raise RuntimeError('Missing final_verification.json: completion cannot be claimed.')
result = json.loads(verification.read_text())
if result.get('state') == 'failed':
    raise RuntimeError(result.get('error', 'Pipeline failed'))
if result.get('fully_completed'):
    required_counts = {'offline_datasets': 2, 'live_questions': 50,
                       'live_generation_calls': 250, 'adaptive_policies': 4}
    for name, count in required_counts.items():
        if result.get(name) != count:
            raise RuntimeError('Incomplete count: ' + name)
    artifacts = result.get('artifacts', {})
    if len(artifacts) < 60 or result.get('artifact_count') != len(artifacts):
        raise RuntimeError('Missing required artifact inventory')
    for filename, expected in artifacts.items():
        path = Path(filename)
        if not path.is_file():
            raise RuntimeError('Missing artifact: ' + filename)
        h = hashlib.sha256()
        with path.open('rb') as f:
            for block in iter(lambda: f.read(8 * 1024 * 1024), b''):
                h.update(block)
        if h.hexdigest() != expected:
            raise RuntimeError('Artifact checksum mismatch: ' + filename)
    log('[COMPLETE] TASK 3 FULLY COMPLETED')
else:
    log('[SAFE STOP] INCOMPLETE — SAVE KAGGLE OUTPUTS AND ATTACH THEM TO RESUME')
log('State: ' + result.get('state', 'unknown'))
log('Offline datasets: ' + str(result.get('offline_datasets', 0)) + '/2')
log('Live questions: ' + str(result.get('live_questions', 0)) + '/50; generation calls: '
    + str(result.get('live_generation_calls', 0)) + '/250')
log('Adaptive policies: ' + str(result.get('adaptive_policies', 0)) + '/4; plots: '
    + str(result.get('plots', 0)) + '; tables: ' + str(result.get('tables', 0))
    + '; total artifacts: ' + str(result.get('artifact_count', 0)))
log('Latest checkpoint: ' + str(result.get('latest_checkpoint', ROOT / 'latest_run.json')))
log('Kaggle output path: ' + str(ROOT))
log('fully_completed = ' + str(result.get('fully_completed', False)))
